# Flat N-Candidate Constraint Solver

## The Problem: Families With More Than Two Products

The original constraint engine (`engine/`) handles **paired products** — one hinge paired with one mounting plate. But some cabinet hardware families require **three or more products** evaluated together:

| Family | Products | Combinations |
|---|---|---|
| LED lighting | Light bar + Driver + Dimmer | A × B × C (triple) |
| Closet systems | Standard + Bracket + Shelf | A × B × C (triple) |
| Drawer systems | Side + Slide + Front fixing | A × B × C (triple) |
| Concealed hinges | Hinge + Plate | A × B (pair) |
| Drawer slides | Slide only | A (single) |

The paired solver (`engine_v2/core/solver.py`) can't express a three-product configuration. This notebook demonstrates **Approach 1: Flat N-Candidate Solver** — a generalization that handles any number of product roles by computing their Cartesian product.

**Reference:** See [`documentation/docs/architecture/multi-family-architecture.md`](../documentation/docs/architecture/multi-family-architecture.md) for the full architectural comparison of both approaches (flat N-candidate vs staged pipeline).

## How It Works

The solver replaces `Configuration(primary, secondary)` with `NConfiguration(candidates: dict[str, Product])`. Products are identified by **role name** rather than position:

```python
config.candidates["light_bar"]   # not config.primary
config.candidates["driver"]      # not config.secondary
config.candidates["dimmer"]      # no slot for this in the paired solver
```

The solve loop computes the **full Cartesian product** of all role lists and evaluates every combination against every rule:

```
For each (bar, driver, dimmer) in bars × drivers × dimmers:
    Evaluate all rules → RuleResults → NConfiguration
    If all rules pass → add to valid list
Sort valid list by ranking criteria
```

## Complexity

**Time:** O(∏ᵢ |Roleᵢ| × R) where R is the number of rules.

For LED lighting with 5 bars × 4 drivers × 4 dimmers × 9 rules = **720 rule evaluations**.

At catalog scale (50 bars × 20 drivers × 30 dimmers × 9 rules) = **270,000 rule evaluations**.

Early termination helps — if the first rule fails, remaining rules are skipped — but every combination is still visited. **There is no pruning between roles.** A bar-driver pair that fails voltage match is re-tested with every dimmer.

This is the key trade-off: simplicity at the cost of redundant work.

In [ ]:
import sys, time
from pathlib import Path
from itertools import product as cartesian_product

# Find project root (works regardless of kernel working directory)
_project_root = Path.cwd()
while _project_root != _project_root.parent:
    if (_project_root / "sample-data").exists() and (_project_root / "engine_v2").exists():
        break
    _project_root = _project_root.parent
sys.path.insert(0, str(_project_root))
DATA_DIR = _project_root / "sample-data"

from engine_v2.core.solver_n import NCandidateSolver, NFamilyConfig
from engine_v2.families.led_lighting.models import (
    LightBar, Driver, Dimmer, LightingRequirements,
    Voltage, DimmingProtocol, ConnectorType,
)
from engine_v2.families.led_lighting.rules import ALL_RULES
from engine_v2.families.led_lighting.test_data import (
    ALL_BARS, ALL_DRIVERS, ALL_DIMMERS,
    BAR_12V_5W, BAR_12V_10W, BAR_24V_15W, BAR_24V_8W_DIM, BAR_12V_LONG,
    DRV_12V_30W, DRV_12V_15W_NODIM, DRV_24V_60W, DRV_24V_20W,
    DIM_TRAILING_150W, DIM_0_10V_200W, DIM_TRAILING_SMALL, DIM_LEADING,
)

# Configure the N-candidate family
led_config = NFamilyConfig(
    name="led_lighting",
    roles=[
        ("light_bar", LightBar),
        ("driver", Driver),
        ("dimmer", Dimmer),
    ],
    requirements_type=LightingRequirements,
    rules=ALL_RULES,
    rank_key=lambda c: (0 if c.total_price_usd is not None else 1, c.total_price_usd or 0),
    early_termination=True,
)

solver = NCandidateSolver(
    config=led_config,
    product_lists={
        "light_bar": ALL_BARS,
        "driver": ALL_DRIVERS,
        "dimmer": ALL_DIMMERS,
    },
)

total_combos = len(ALL_BARS) * len(ALL_DRIVERS) * len(ALL_DIMMERS)
print(f"Catalog: {len(ALL_BARS)} light bars, {len(ALL_DRIVERS)} drivers, {len(ALL_DIMMERS)} dimmers")
print(f"Cartesian product: {len(ALL_BARS)} x {len(ALL_DRIVERS)} x {len(ALL_DIMMERS)} = {total_combos} combinations")
print(f"Rules per combination: {len(ALL_RULES)}")
print(f"Max rule evaluations (no early termination): {total_combos * len(ALL_RULES)}")
print(f"\nFamily config: {len(led_config.roles)} roles, {len(led_config.rules)} rules")

## 1. The LED Lighting Domain

A cabinet LED lighting system consists of three products that must be electrically and physically compatible:

**Light bar** — the LED strip mounted inside the cabinet. Defines voltage (12V/24V DC), wattage, lumen output, connector type, and whether it supports dimming.

**Driver (transformer)** — converts mains AC to the bar's DC voltage. Must match the bar's voltage, have enough wattage capacity (with 20% headroom for longevity), and support dimming if required.

**Dimmer switch** — optional brightness control. Must use the same dimming protocol as the driver (trailing-edge, leading-edge, 0-10V, or DALI), handle the system wattage, and be voltage-compatible.

### Constraint relationships

```
Light bar <-> Driver:    Voltage match, wattage capacity, connector type
Light bar <-> Dimmer:    (no direct constraints)
Driver    <-> Dimmer:    Dimming protocol, voltage compatibility
All three:               Total wattage within dimmer range
```

This is important for understanding why the staged pipeline approach (see `demo/v2_staged_pipeline_demo.ipynb`) can be more efficient — the bar-dimmer relationship has **no direct constraints**, meaning bar-driver failures can be pruned without ever considering dimmers.

### Rules (9 total)

| Rule | Products involved | Constraint |
|---|---|---|
| LED001 | Bar + Driver | Voltage must match |
| LED002 | Bar + Driver | Total wattage within driver's 80% safe capacity |
| LED003 | Bar + Driver | Connector type must match |
| LED006 | Bar + Cabinet | Bar length must fit cabinet |
| LED007 | Bar | Meets minimum brightness requirement |
| LED008 | Driver | Supports dimming if required |
| LED004 | Driver + Dimmer | Dimming protocol must match |
| LED005 | Bar + Dimmer | Total wattage within dimmer's min/max range |
| LED009 | Driver + Dimmer | Dimmer voltage compatibility |

See `engine_v2/families/led_lighting/rules.py` for implementations.

In [ ]:
# Display the product catalog
print("=" * 90)
print("LIGHT BARS")
print("=" * 90)
print(f"{'SKU':<22} {'Brand':<8} {'Voltage':<8} {'Watts':>6} {'Lumens':>7} {'Length':>7} {'Dimmable':<9} {'Connector':<16} {'Price':>7}")
print("-" * 90)
for b in ALL_BARS:
    print(f"{b.sku:<22} {b.brand:<8} {b.voltage.value:<8} {b.wattage:>5.0f}W {b.lumen_output:>6}lm {b.length_mm:>5}mm {'Yes' if b.dimmable else 'No':<9} {b.connector.value:<16} ${b.price_usd:>6.2f}")

print(f"\n{'=' * 90}")
print("DRIVERS")
print("=" * 90)
print(f"{'SKU':<22} {'Brand':<8} {'Voltage':<8} {'Max W':>6} {'Channels':>9} {'Dimmable':<9} {'Protocol':<16} {'Connector':<16} {'Price':>7}")
print("-" * 90)
for d in ALL_DRIVERS:
    print(f"{d.sku:<22} {d.brand:<8} {d.output_voltage.value:<8} {d.max_wattage:>5.0f}W {d.output_channels:>9} {'Yes' if d.dimmable else 'No':<9} {d.dimming_protocol.value:<16} {d.connector.value:<16} ${d.price_usd:>6.2f}")

print(f"\n{'=' * 90}")
print("DIMMERS")
print("=" * 90)
print(f"{'SKU':<22} {'Brand':<8} {'Protocol':<16} {'Max W':>6} {'Min W':>6} {'Voltages':<16} {'Price':>7}")
print("-" * 90)
for d in ALL_DIMMERS:
    volts = ", ".join(v.value for v in d.voltage_compatible)
    print(f"{d.sku:<22} {d.brand:<8} {d.dimming_protocol.value:<16} {d.max_wattage:>5.0f}W {d.min_load_wattage:>5.0f}W {volts:<16} ${d.price_usd:>6.2f}")

## 2. Solving a Lighting Scenario

The solver evaluates every (bar, driver, dimmer) triple against all 9 rules. With early termination enabled, a voltage mismatch at LED001 skips the remaining 8 rules for that triple, but the triple is still visited.

Let's solve a common scenario: a 600mm under-cabinet light with dimming.

In [ ]:
# Scenario: 600mm cabinet, dimming required, no brand preference
req = LightingRequirements(
    cabinet_length_mm=600,
    dimming_required=True,
    min_lumen_output=300,
)

result = solver.solve_with_explanation(req)

print(f"Status: {result['status']}")
print(f"{result['message']}\n")

if result["status"] == "solved":
    rec = result["recommended"]
    print("RECOMMENDED CONFIGURATION:")
    print(f"  Light bar: {rec['candidates']['light_bar']['sku']}")
    print(f"  Driver:    {rec['candidates']['driver']['sku']}")
    print(f"  Dimmer:    {rec['candidates']['dimmer']['sku']}")
    price = f"${rec['total_price_usd']:.2f}" if rec['total_price_usd'] else "N/A"
    print(f"  Total price: {price}")
    print(f"\n  Constraint trace ({len(rec['constraint_trace'])} rules):")
    for rule in rec["constraint_trace"]:
        icon = "PASS" if rule["passed"] else "FAIL"
        print(f"    [{icon}] {rule['rule']} {rule['name']}: {rule['detail']}")
    
    print(f"\n  + {len(result['alternatives'])} alternative(s)")

In [ ]:
# Show all valid configurations with pricing
valid = solver.solve(req)

print(f"All {len(valid)} valid configurations:\n")
print(f"{'#':>3}  {'Light Bar':<22} {'Driver':<22} {'Dimmer':<22} {'Total Price':>11}")
print("-" * 85)
for i, config in enumerate(valid, 1):
    bar = config.candidates["light_bar"]
    drv = config.candidates["driver"]
    dim = config.candidates["dimmer"]
    price = f"${config.total_price_usd:.2f}" if config.total_price_usd else "N/A"
    print(f"{i:>3}  {bar.sku:<22} {drv.sku:<22} {dim.sku:<22} {price:>11}")

## 3. Failure Analysis

When no valid configuration exists, the solver finds the **closest match** — the combination that passes the most rules — and reports exactly which rules failed. This is the same explainability pattern used in the hinge engine (see `demo/v1_hinge_constraint_demo.ipynb`).

Here we use a 200mm cabinet — shorter than any light bar in the catalog (minimum 300mm) — to guarantee no valid configuration exists.

In [ ]:
# Impossible scenario: cabinet too small for any light bar (shortest is 300mm)
fail_req = LightingRequirements(
    cabinet_length_mm=200,
    dimming_required=True,
    min_lumen_output=300,
)

fail_result = solver.solve_with_explanation(fail_req)

print(f"Status: {fail_result['status']}")
print(f"{fail_result['message']}\n")

if fail_result["status"] == "no_solution":
    if fail_result.get("closest_match"):
        cm = fail_result["closest_match"]
        print("Closest match:")
        for role, prod in cm["candidates"].items():
            print(f"  {role}: {prod['sku']}")

    print(f"\nFailed constraints:")
    for fr in fail_result["failed_rules"]:
        print(f"  [{fr['rule']}] {fr['name']}: {fr['detail']}")
        if fr.get("remediation"):
            print(f"           Fix: {fr['remediation']}")
else:
    print(f"Unexpectedly found {len(fail_result.get('alternatives', [])) + 1} valid configuration(s)")

## 4. Exhaustive Evaluation Matrix

The N-candidate solver evaluates **every** triple. Let's visualize this — how many of the 80 combinations pass all rules, and where do the failures concentrate?

This is the core visibility advantage of the flat approach: because every combination is evaluated, we can build a complete picture of the constraint space.

In [ ]:
# Evaluate every triple (disable early termination for full traces)
led_config.early_termination = False

baseline_req = LightingRequirements(cabinet_length_mm=800)

total = 0
valid_count = 0
fail_counts = {}  # rule_id -> count of failures

for bar in ALL_BARS:
    for drv in ALL_DRIVERS:
        for dim in ALL_DIMMERS:
            total += 1
            candidates = {"light_bar": bar, "driver": drv, "dimmer": dim}
            config = solver.evaluate(candidates, baseline_req)
            if config.valid:
                valid_count += 1
            else:
                for r in config.failed_rules:
                    fail_counts[r.rule_id] = fail_counts.get(r.rule_id, 0) + 1

led_config.early_termination = True  # Restore

print(f"Total combinations evaluated: {total}")
print(f"Valid configurations: {valid_count} ({valid_count/total*100:.1f}%)")
print(f"Invalid configurations: {total - valid_count} ({(total-valid_count)/total*100:.1f}%)")
print(f"\nFailure breakdown by rule:")
print(f"{'Rule':<10} {'Name':<28} {'Failures':>8} {'% of invalid':>12}")
print("-" * 60)
for rule_id, count in sorted(fail_counts.items(), key=lambda x: -x[1]):
    # Look up rule name
    name = rule_id
    for r in ALL_RULES:
        test_result = r({"light_bar": ALL_BARS[0], "driver": ALL_DRIVERS[0], "dimmer": ALL_DIMMERS[0]}, baseline_req, {})
        if test_result.rule_id == rule_id:
            name = test_result.rule_name
            break
    pct = count / (total - valid_count) * 100 if (total - valid_count) > 0 else 0
    print(f"{rule_id:<10} {name:<28} {count:>8} {pct:>11.1f}%")

print(f"\nNote: Percentages exceed 100% because a single combination can fail multiple rules.")

## 5. Scaling Analysis — The Combinatorial Problem

The flat N-candidate approach computes the **full Cartesian product**. This is fine for small catalogs, but grows multiplicatively with each role's catalog size.

The table below projects the evaluation counts for realistic catalog sizes. Compare these numbers with the staged pipeline approach in `demo/v2_staged_pipeline_demo.ipynb`, which prunes invalid partial combinations between stages.

**Reference:** See the "Comparison: N-candidate vs Staged" table in [`documentation/docs/architecture/multi-family-architecture.md`](../documentation/docs/architecture/multi-family-architecture.md) for the full trade-off analysis.

In [ ]:
# Scaling projections
import random

num_rules = len(ALL_RULES)

print("SCALING PROJECTIONS — Flat N-Candidate Solver")
print("=" * 90)
print(f"{'Scenario':<30} {'Bars':>6} {'Drivers':>8} {'Dimmers':>8} {'Combos':>12} {'Max Rule Evals':>15}")
print("-" * 90)

scenarios = [
    ("Current prototype",        5,     4,      4),
    ("Small distributor",       20,    10,     10),
    ("Medium catalog",          50,    20,     30),
    ("Full Wurth catalog",     100,    40,     50),
    ("Multi-brand aggregate",  200,    80,    100),
]

for name, bars, drivers, dimmers in scenarios:
    combos = bars * drivers * dimmers
    max_evals = combos * num_rules
    print(f"{name:<30} {bars:>6} {drivers:>8} {dimmers:>8} {combos:>12,} {max_evals:>15,}")

print(f"\n{'=' * 90}")
print("COMPLEXITY GROWTH")
print("=" * 90)
print(f"\nWith 3 roles, complexity is O(A x B x C x R)")
print(f"  - Doubling ANY one role doubles the total work")
print(f"  - Doubling ALL three roles increases work by 8x (2^3)")
print(f"  - Adding a 4th role (e.g., mounting clip) multiplies by |D|")
print(f"\nFor a 4-product family (A x B x C x D):")
for name, a, b, c in scenarios[2:]:
    d = 15  # hypothetical 4th role
    combos_4 = a * b * c * d
    print(f"  {name}: {a} x {b} x {c} x {d} = {combos_4:,} combinations")

In [ ]:
# Benchmark: measure actual time with early termination on the current catalog
# Then project what a larger catalog would cost

def generate_synthetic_bars(n, template=BAR_12V_5W):
    """Generate n synthetic light bars for benchmarking."""
    bars = []
    voltages = [Voltage.DC_12V, Voltage.DC_24V]
    connectors = [ConnectorType.BARREL_JACK, ConnectorType.TERMINAL_BLOCK]
    for i in range(n):
        bars.append(LightBar(
            sku=f"BAR-SYNTH-{i:04d}",
            brand=random.choice(["Loox", "Hafele", "Domus"]),
            price_usd=round(random.uniform(10, 80), 2),
            wattage=random.choice([5, 8, 10, 15, 20]),
            voltage=random.choice(voltages),
            length_mm=random.choice([300, 450, 600, 900, 1200]),
            lumen_output=random.choice([300, 500, 800, 1200, 1600]),
            dimmable=random.choice([True, True, False]),  # 67% dimmable
            connector=random.choice(connectors),
        ))
    return bars

def generate_synthetic_drivers(n):
    drivers = []
    for i in range(n):
        v = random.choice([Voltage.DC_12V, Voltage.DC_24V])
        conn = ConnectorType.BARREL_JACK if v == Voltage.DC_12V else ConnectorType.TERMINAL_BLOCK
        dimmable = random.choice([True, True, False])
        proto = random.choice([DimmingProtocol.TRAILING_EDGE, DimmingProtocol.ZERO_TO_10V]) if dimmable else DimmingProtocol.NONE
        drivers.append(Driver(
            sku=f"DRV-SYNTH-{i:04d}",
            brand=random.choice(["Loox", "Hafele", "Mean Well"]),
            price_usd=round(random.uniform(10, 60), 2),
            output_voltage=v,
            max_wattage=random.choice([15, 30, 60, 100]),
            output_channels=random.choice([2, 4, 6]),
            dimmable=dimmable,
            dimming_protocol=proto,
            connector=conn,
        ))
    return drivers

def generate_synthetic_dimmers(n):
    dimmers = []
    for i in range(n):
        proto = random.choice([DimmingProtocol.TRAILING_EDGE, DimmingProtocol.ZERO_TO_10V, DimmingProtocol.LEADING_EDGE])
        dimmers.append(Dimmer(
            sku=f"DIM-SYNTH-{i:04d}",
            brand=random.choice(["Loox", "Hafele", "Lutron"]),
            price_usd=round(random.uniform(15, 80), 2),
            dimming_protocol=proto,
            max_wattage=random.choice([50, 100, 150, 200]),
            voltage_compatible=random.choice([[Voltage.DC_12V], [Voltage.DC_24V], [Voltage.DC_12V, Voltage.DC_24V]]),
            min_load_wattage=random.choice([0, 2, 5, 10]),
        ))
    return dimmers

# Benchmark at different scales
print("BENCHMARK — Flat N-Candidate Solver (early termination ON)")
print("=" * 90)
print(f"{'Bars':>6} {'Drivers':>8} {'Dimmers':>8} {'Combos':>10} {'Time (ms)':>10} {'Combos/sec':>12} {'Valid':>6}")
print("-" * 90)

random.seed(42)
benchmark_scenarios = [
    (5,    4,    4),
    (20,   10,   10),
    (50,   20,   30),
    (100,  40,   50),
]

req = LightingRequirements(cabinet_length_mm=600, dimming_required=True)

for n_bars, n_drivers, n_dimmers in benchmark_scenarios:
    bars = generate_synthetic_bars(n_bars) if n_bars > 5 else ALL_BARS
    drivers = generate_synthetic_drivers(n_drivers) if n_drivers > 4 else ALL_DRIVERS
    dimmers = generate_synthetic_dimmers(n_dimmers) if n_dimmers > 4 else ALL_DIMMERS
    
    bench_solver = NCandidateSolver(
        config=led_config,
        product_lists={"light_bar": bars, "driver": drivers, "dimmer": dimmers},
    )
    
    start = time.perf_counter()
    results = bench_solver.solve(req)
    elapsed_ms = (time.perf_counter() - start) * 1000
    
    combos = n_bars * n_drivers * n_dimmers
    rate = combos / (elapsed_ms / 1000) if elapsed_ms > 0 else 0
    print(f"{n_bars:>6} {n_drivers:>8} {n_dimmers:>8} {combos:>10,} {elapsed_ms:>9.1f}ms {rate:>11,.0f} {len(results):>6}")

print(f"\nNote: Synthetic products are randomly generated. Valid counts vary by run.")

## 6. The Redundancy Problem

The key weakness of the flat approach: **no inter-role pruning**. If a 12V bar is incompatible with a 24V driver (LED001 fails), the flat solver still evaluates that pair with every dimmer. With 30 dimmers, that's 30 wasted evaluations for a pair that was already known to be invalid.

The cell below quantifies this waste by counting how many triples fail on bar-driver rules alone (LED001-003, 006-008) — these failures have nothing to do with the dimmer and would be pruned at Stage 1 in the staged pipeline approach.

In [ ]:
# Quantify redundant work: how many triples fail on bar-driver rules?
from engine_v2.families.led_lighting.rules import STAGE_1_RULES, STAGE_2_RULES

baseline_req = LightingRequirements(cabinet_length_mm=800)

# First: evaluate bar x driver pairs (without dimmer)
bar_driver_pairs = len(ALL_BARS) * len(ALL_DRIVERS)
valid_pairs = 0
invalid_pairs = 0

for bar in ALL_BARS:
    for drv in ALL_DRIVERS:
        # Evaluate only stage 1 rules
        pair_valid = True
        for rule in STAGE_1_RULES:
            candidates = {"light_bar": bar, "driver": drv, "dimmer": ALL_DIMMERS[0]}  # dimmer unused by stage 1
            result = rule(candidates, baseline_req, {})
            if not result.passed and result.category.value == "hard_constraint":
                pair_valid = False
                break
        if pair_valid:
            valid_pairs += 1
        else:
            invalid_pairs += 1

n_dimmers = len(ALL_DIMMERS)
total_triples = bar_driver_pairs * n_dimmers
wasted_triples = invalid_pairs * n_dimmers

print("REDUNDANCY ANALYSIS")
print("=" * 70)
print(f"Bar x Driver pairs:     {bar_driver_pairs}")
print(f"  Valid pairs:           {valid_pairs}")
print(f"  Invalid pairs:         {invalid_pairs}")
print(f"")
print(f"Total triples (flat):    {total_triples}")
print(f"  Triples from valid pairs:   {valid_pairs * n_dimmers}")
print(f"  Triples from invalid pairs: {wasted_triples}  <-- WASTED WORK")
print(f"")
print(f"Waste ratio: {wasted_triples/total_triples*100:.1f}% of all triples are redundant")
print(f"")
print(f"The staged pipeline (demo/v2_staged_pipeline_demo.ipynb) eliminates")
print(f"this waste by pruning invalid bar-driver pairs BEFORE dimmer evaluation.")
print(f"It would evaluate {bar_driver_pairs} pairs + {valid_pairs * n_dimmers} triples")
print(f"= {bar_driver_pairs + valid_pairs * n_dimmers} total evaluations vs {total_triples} in the flat approach.")

## 7. When to Use This Approach

The flat N-candidate solver is the right choice when:

- **All products constrain each other.** If every product has rules involving every other product, there's nothing to prune between stages. The staged approach adds complexity without benefit.
- **Catalogs are small.** Under ~10,000 combinations, the flat approach completes in milliseconds. Simplicity wins.
- **You need the full evaluation matrix.** Analytics, compatibility reports, and coverage analysis all benefit from exhaustive evaluation.
- **You're prototyping.** One loop, one rule list, no stage ordering to debug.

It is **not** the right choice when:

- **Catalogs are large and constraints are layered.** 100 bars × 40 drivers × 50 dimmers = 200,000 combinations. If 80% of bar-driver pairs fail, the staged approach evaluates ~42,000 instead.
- **Adding a 4th product role.** The flat approach grows by a factor of |D|. The staged approach adds one more pruning stage.
- **Latency matters.** An API endpoint serving real-time recommendations shouldn't compute 200K+ combinations per request.

**For the comparison, see:** `demo/v2_staged_pipeline_demo.ipynb`

**For the full architecture discussion, see:** [`documentation/docs/architecture/multi-family-architecture.md`](../documentation/docs/architecture/multi-family-architecture.md)

**Source code:** `engine_v2/core/solver_n.py` (solver), `engine_v2/families/led_lighting/` (models + rules), `engine_v2/tests/test_n_candidate.py` (26 tests)